# Caso "Cancelado" — Análisis de Confiabilidad de Evidencia
## Kai Duarte: separando la señal real del ruido fabricado

**Contexto:** Un clip de audio de 14 segundos con una voz que "suena a" Kai Duarte admitiendo amañar un sorteo viraliza 12.000 veces en 40 minutos. Tres marcas patrocinadoras evalúan romper sus contratos. Nuestra misión: determinar qué tan confiable es cada pieza de evidencia y recomendar una de tres acciones: **Cancelar · Bajar el video · Mantener**.

---
**Dataset:** 3.210 publicaciones sintéticas, 17 variables, ventana temporal 11:47 PM – 7:30 AM (8 h)  
**Metodología:** Ingeniería de features → Score de Confiabilidad → Decisión basada en segmentos

> _Nota: Datos sintéticos, seguros para publicar. Caso ficticio con fines académicos._


In [ ]:
import sys
import json
import warnings
warnings.filterwarnings("ignore")

sys.path.insert(0, "src")

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from features import (
    aplicar_pipeline_completo,
    calcular_porcentajes_decision,
    UMBRAL_CANCELAR,
    UMBRAL_BAJAR,
)

# Paleta Light BI
AZUL = "#1e6cbf"; AZUL_CLARO = "#4a9edd"
ROJO = "#d9534f"; AMARILLO = "#f0a500"; VERDE = "#3aaa5c"
GRIS_FONDO = "#f7f9fc"; GRIS_GRID = "#e0e6ed"

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.facecolor": GRIS_FONDO,
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.color": GRIS_GRID,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

print("Librerías cargadas OK")


## 1. Carga y limpieza de datos

In [ ]:
# Cargar CSV crudo
df_raw = pd.read_csv("data/BASE.csv")

print(f"Registros totales  : {len(df_raw):,}")
print(f"Columnas           : {len(df_raw.columns)}")
print()

# Renombrar columna con tilde para compatibilidad ASCII
df_raw.rename(columns={"Antig\u00fcedad_Cuenta_Dias": "Antiguedad_Cuenta_Dias"}, inplace=True)

# Valores faltantes
nulos = df_raw.isnull().sum()
nulos_pct = (nulos / len(df_raw) * 100).round(1)
display(pd.DataFrame({"Nulos": nulos, "Porcentaje %": nulos_pct})[nulos > 0])


In [ ]:
# Imputación de valores faltantes
df = df_raw.copy()

# Antigüedad y Interacciones: imputar con mediana (cuentas en zona gris)
df["Antiguedad_Cuenta_Dias"].fillna(df["Antiguedad_Cuenta_Dias"].median(), inplace=True)
df["Num_Interacciones"].fillna(df["Num_Interacciones"].median(), inplace=True)

# Campos de texto: NaN es válido
df["Hashtags_Usados"].fillna("(sin hashtags)", inplace=True)
df["Marca_Mencionada"].fillna("(sin mención)", inplace=True)

df["Fecha"] = pd.to_datetime(df["Fecha_Hora_Publicacion"])

print("Limpieza completada. Nulos restantes:", df.isnull().sum().sum())
print(f"Período: {df.Fecha.min()} → {df.Fecha.max()}")


## 2. Análisis exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Plataforma
plat_counts = df["Plataforma"].value_counts()
axes[0].bar(plat_counts.index, plat_counts.values,
            color=[AZUL, AZUL_CLARO, AMARILLO, GRIS_GRID], edgecolor="white")
axes[0].set_title("Publicaciones por plataforma", fontweight="bold")
axes[0].set_ylabel("Publicaciones")
for i, v in enumerate(plat_counts.values):
    axes[0].text(i, v+10, str(v), ha="center", fontsize=9)

# Tipo de publicación
tipo_counts = df["Tipo_Publicacion"].value_counts()
labels_c = [t.replace("Transcripción de audio/video","Trans. A/V")
              .replace("Descripción de captura","Desc. captura")
            for t in tipo_counts.index]
axes[1].barh(labels_c[::-1], tipo_counts.values[::-1], color=AZUL_CLARO, edgecolor="white", alpha=0.85)
axes[1].set_title("Tipo de publicación", fontweight="bold")
axes[1].set_xlabel("Publicaciones")

# Perfil de usuario
perfil_counts = df["Perfil_Usuario"].value_counts()
colores_p = {"Bot_Sospechoso":ROJO,"Cuenta_Nueva":AMARILLO,
             "An\u00f3nimo":"#f5a623","Fan_Antiguo":VERDE,"Verificado":AZUL}
cp = [colores_p.get(p, GRIS_GRID) for p in perfil_counts.index]
axes[2].bar(perfil_counts.index, perfil_counts.values, color=cp, edgecolor="white", alpha=0.85)
axes[2].set_title("Perfil de usuario", fontweight="bold")
axes[2].tick_params(axis="x", rotation=25)
for i, v in enumerate(perfil_counts.values):
    axes[2].text(i, v+10, str(v), ha="center", fontsize=9)

plt.suptitle("Distribuciones clave del dataset", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figuras/eda_distribuciones.png", dpi=130, bbox_inches="tight")
plt.show()

bot_nueva = perfil_counts.get("Bot_Sospechoso",0) + perfil_counts.get("Cuenta_Nueva",0)
print(f"Cuentas Bot+Nueva: {bot_nueva:,} ({bot_nueva/len(df)*100:.1f}%)")
print(f"Contenido reciclado confirmado: {df.Contenido_Reciclado.str.startswith(chr(83)+"\u00ed").sum()}")


In [ ]:
# Estadísticas descriptivas — variables numéricas
cols_num = ["Antiguedad_Cuenta_Dias","Num_Seguidores_Cuenta",
            "Num_Interacciones","Velocidad_Viralizacion","Corroboraciones_Independientes"]
display(df[cols_num].describe().round(2))

print(f"\nVelocidad > 50 (pico sospechoso): {(df.Velocidad_Viralizacion>50).sum()} ")
print(f"Cuentas < 30 días de antigüedad : {(df.Antiguedad_Cuenta_Dias<30).sum()} ")


## 3. Ingeniería de features

### Fórmulas de las variables derivadas

| Variable | Fórmula | Rango |
|---|---|---|
| **Indice_Sospecha** | 0.30×perfil + 0.25×antigüedad_inv + 0.25×velocidad_pct + 0.20×reciclado | [0, 1] |
| **Tasa_Coordinacion** | % publicaciones sospechosas en ventana ±15 min | [0, 1] |
| **Peso_Prueba** | tipo_base + corrobora×0.04 − penalización_reciclado | [0, 1] |
| **Score_Confiabilidad** | 40×(1−IS) + 35×PP + 15×(1−TC) + 10×corrobora_norm | [0, 100] |

**Umbrales de decisión:**  
- Score < 40 → `Cancelar` (rojo)  
- 40 ≤ Score ≤ 65 → `Bajar el video` (amarillo)  
- Score > 65 → `Mantener` (verde)


In [ ]:
# Aplicar pipeline completo de features
df_e = aplicar_pipeline_completo(df_raw)

print("Columnas nuevas:", [c for c in df_e.columns if c not in df_raw.columns])
print()
display(df_e[["Indice_Sospecha","Tasa_Coordinacion","Peso_Prueba","Score_Confiabilidad"]].describe().round(3))


In [ ]:
# Mapa de correlaciones
df_e2 = df_e.copy()
df_e2.rename(columns={"Antig\u00fcedad_Cuenta_Dias":"Antiguedad_Cuenta_Dias"}, errors="ignore", inplace=True)
df_e2["perfil_num"] = df_e2["Perfil_Usuario"].map(
    {"Bot_Sospechoso":5,"Cuenta_Nueva":4,"An\u00f3nimo":3,"Fan_Antiguo":2,"Verificado":1})

corr_cols = ["Antiguedad_Cuenta_Dias","Velocidad_Viralizacion","Corroboraciones_Independientes",
             "perfil_num","Indice_Sospecha","Peso_Prueba","Score_Confiabilidad"]
corr = df_e2[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr.values, cmap="RdYlGn", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(corr_cols))); ax.set_yticks(range(len(corr_cols)))
labels_corr = ["Antigüedad","Velocidad","Corroboraciones","Perfil","IS","PP","SC"]
ax.set_xticklabels(labels_corr, fontsize=8, rotation=30)
ax.set_yticklabels(labels_corr, fontsize=8)
for i in range(len(corr_cols)):
    for j in range(len(corr_cols)):
        ax.text(j, i, str(corr.values[i,j]), ha="center", va="center", fontsize=7,
                color="white" if abs(corr.values[i,j]) > 0.5 else "black")
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title("Correlaciones — features originales vs. derivadas", fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figuras/correlaciones.png", dpi=130, bbox_inches="tight")
plt.show()


## 4. Clustering K-Means: grupos naturales de publicaciones

Aplicamos K-Means sobre las 4 features derivadas para descubrir patrones sin necesidad de etiquetas supervisadas.


In [ ]:
# Preparar features escaladas
feat_cluster = ["Indice_Sospecha","Tasa_Coordinacion","Peso_Prueba","Score_Confiabilidad"]
X = df_e[feat_cluster].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Selección de K
inercias = []; silhouettes = []
K_range = range(2, 7)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
ax1.plot(list(K_range), inercias, "o-", color=AZUL)
ax1.set_xlabel("K"); ax1.set_ylabel("Inercia"); ax1.set_title("Método del codo", fontweight="bold")
ax2.plot(list(K_range), silhouettes, "s-", color=VERDE)
ax2.set_xlabel("K"); ax2.set_ylabel("Silhouette"); ax2.set_title("Silhouette", fontweight="bold")
plt.suptitle("Selección de K para clustering", fontweight="bold")
plt.tight_layout(); plt.show()
k_opt = list(K_range)[silhouettes.index(max(silhouettes))]
print(f"K óptimo: {k_opt}  |  Silhouette: {max(silhouettes):.3f}")


In [ ]:
# Aplicar K-Means con K=3 (alineado con las 3 zonas de decisión)
km_final = KMeans(n_clusters=3, random_state=42, n_init=10)
df_e["Cluster"] = km_final.fit_predict(X_scaled)

cluster_desc = df_e.groupby("Cluster").agg(
    tama\u00f1o=("ID_Publicacion","count"),
    score_prom=("Score_Confiabilidad","mean"),
    sospecha_prom=("Indice_Sospecha","mean"),
    perfil_modal=("Perfil_Usuario", lambda x: x.mode()[0]),
).round(2)
display(cluster_desc)

# Etiquetar clusters por score promedio
scores = cluster_desc["score_prom"].to_dict()
ordenados = sorted(scores.items(), key=lambda x: x[1])
etiquetas = {ordenados[0][0]:"Alta sospecha", ordenados[1][0]:"Sospecha media", ordenados[2][0]:"Baja sospecha"}
df_e["Cluster_Label"] = df_e["Cluster"].map(etiquetas)
print("\nDistribución:")
print(df_e["Cluster_Label"].value_counts())


In [ ]:
# Visualizar clusters
fig, ax = plt.subplots(figsize=(9, 6))
colores_c = {"Alta sospecha":ROJO, "Sospecha media":AMARILLO, "Baja sospecha":VERDE}
for lbl, grupo in df_e.groupby("Cluster_Label"):
    ax.scatter(grupo["Indice_Sospecha"], grupo["Score_Confiabilidad"],
               c=colores_c[lbl], alpha=0.3, s=15, label=lbl)
ax.set_xlabel("Índice de Sospecha"); ax.set_ylabel("Score de Confiabilidad")
ax.set_title("Clusters: Sospecha vs. Confiabilidad", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig("outputs/figuras/clusters_scatter.png", dpi=130, bbox_inches="tight")
plt.show()


## 5. Score de Confiabilidad y decisión final

In [ ]:
# Histograma semáforo
fig, ax = plt.subplots(figsize=(9, 5))
bins = range(0, 105, 5)
n, _, patches = ax.hist(df_e["Score_Confiabilidad"], bins=bins, edgecolor="white", linewidth=0.5)
for patch in patches:
    c = patch.get_x()
    patch.set_facecolor(ROJO if c < 40 else (AMARILLO if c < 65 else VERDE))
    patch.set_alpha(0.85)
ax.axvline(40, color=ROJO,  linestyle="--", linewidth=1.5)
ax.axvline(65, color=VERDE, linestyle="--", linewidth=1.5)
ax.set_xlabel("Score de Confiabilidad"); ax.set_ylabel("Publicaciones")
ax.set_title("Distribución del Score de Confiabilidad\nCaso Kai Duarte", fontsize=13, fontweight="bold")
c_n = (df_e["Score_Confiabilidad"]<40).sum()
b_n = ((df_e["Score_Confiabilidad"]>=40)&(df_e["Score_Confiabilidad"]<=65)).sum()
m_n = (df_e["Score_Confiabilidad"]>65).sum()
ax.legend(handles=[
    mpatches.Patch(color=ROJO,    alpha=0.85, label=f"Cancelar (<40) — {c_n} pub."),
    mpatches.Patch(color=AMARILLO, alpha=0.85, label=f"Bajar video (40–65) — {b_n} pub."),
    mpatches.Patch(color=VERDE,   alpha=0.85, label=f"Mantener (>65) — {m_n} pub."),
], loc="upper right", fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figuras/fig1_score_confiabilidad.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
pct = calcular_porcentajes_decision(df_e["Decision_Semaforo"])
sep = "=" * 52
print(sep)
print("  RESULTADO — CASO KAI DUARTE")
print(sep)
print(f"  Cancelar       : {pct['Cancelar']:5.1f} %")
print(f"  Bajar el video : {pct['Bajar el video']:5.1f} %")
print(f"  Mantener       : {pct['Mantener']:5.1f} %")
print("-" * 52)
print(f"  DECISIÓN FINAL : {pct['decision_final'].upper()}")
print(f"  CONFIANZA      : {pct['confianza_pct']:.1f} %")
print(sep)


## 6. Línea de tiempo del escándalo

In [ ]:
df_e["minuto"] = pd.to_datetime(df_e["Fecha_Hora_Publicacion"]).dt.floor("5min")
tl = df_e.groupby("minuto").agg(
    total=("ID_Publicacion","count"),
    cancelar=("Decision_Semaforo", lambda x:(x=="Cancelar").sum()),
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(tl["minuto"], tl["total"], alpha=0.2, color=AZUL)
ax.plot(tl["minuto"], tl["total"], color=AZUL, linewidth=2, label="Total publicaciones")
ax.fill_between(tl["minuto"], tl["cancelar"], alpha=0.35, color=ROJO)
ax.plot(tl["minuto"], tl["cancelar"], color=ROJO, linewidth=1.5, linestyle="--", label="Zona Cancelar")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
plt.xticks(rotation=30)
ax.set_xlabel("Hora (2026-03-15)"); ax.set_ylabel("Publicaciones / 5 min")
ax.set_title("Línea de tiempo del escándalo", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("outputs/figuras/fig2_linea_tiempo.png", dpi=150, bbox_inches="tight")
plt.show()

pico = tl.loc[tl["total"].idxmax()]
print(f"Pico de actividad: {pico['minuto'].strftime('%H:%M')} — {pico['total']} publicaciones en 5 min")


## 7. Índice de Sospecha por plataforma y perfil

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))

plat = df_e.groupby("Plataforma")["Indice_Sospecha"].mean().sort_values()
cp = [ROJO if v>0.45 else AMARILLO if v>0.35 else VERDE for v in plat.values]
bars = axes[0].barh(plat.index, plat.values, color=cp, alpha=0.85, edgecolor="white")
axes[0].set_xlabel("Índice de Sospecha prom."); axes[0].set_title("Por Plataforma", fontweight="bold")
axes[0].set_xlim(0, 0.75)
for bar, val in zip(bars, plat.values): axes[0].text(val+0.01,bar.get_y()+bar.get_height()/2,f"{val:.2f}",va="center",fontsize=9)

perfil = df_e.groupby("Perfil_Usuario")["Indice_Sospecha"].mean().sort_values()
cp2 = [ROJO if v>0.5 else AMARILLO if v>0.35 else VERDE for v in perfil.values]
bars2 = axes[1].barh(perfil.index, perfil.values, color=cp2, alpha=0.85, edgecolor="white")
axes[1].set_xlabel("Índice de Sospecha prom."); axes[1].set_title("Por Perfil", fontweight="bold")
axes[1].set_xlim(0, 0.95)
for bar, val in zip(bars2, perfil.values): axes[1].text(val+0.01,bar.get_y()+bar.get_height()/2,f"{val:.2f}",va="center",fontsize=9)

fig.suptitle("Índice de Sospecha por Plataforma y Perfil", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("outputs/figuras/fig3_sospecha_plataforma_perfil.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Ranking de cuentas más sospechosas

In [ ]:
top_c = (
    df_e.groupby("Usuario_Handle").agg(
        indice_sospecha=("Indice_Sospecha","mean"),
        score_prom=("Score_Confiabilidad","mean"),
        num_pub=("ID_Publicacion","count"),
        perfil=("Perfil_Usuario", lambda x: x.mode()[0]),
    )
    .query("num_pub >= 2")
    .sort_values("indice_sospecha", ascending=False)
    .head(20).round(2)
)
display(top_c)


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
ct = [ROJO if v>0.65 else AMARILLO if v>0.5 else AZUL_CLARO for v in top_c["indice_sospecha"]]
bars3 = ax.barh(top_c.index[::-1], top_c["indice_sospecha"][::-1], color=ct[::-1], alpha=0.85, edgecolor="white")
ax.set_xlabel("Índice de Sospecha prom."); ax.set_title("Top 20 cuentas más sospechosas", fontweight="bold")
ax.set_xlim(0, 1.0)
for bar, val in zip(bars3, top_c["indice_sospecha"][::-1]): ax.text(val+0.01,bar.get_y()+bar.get_height()/2,f"{val:.2f}",va="center",fontsize=8)
plt.tight_layout()
plt.savefig("outputs/figuras/fig4_top_cuentas_sospechosas.png", dpi=150, bbox_inches="tight")
plt.show()


## 9. Exportar resultados para el dashboard

In [ ]:
# Ejecutar el script de exportación del JSON
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-c",
     "import sys; sys.path.insert(0,'src'); exec(open('crear_notebook.py').read())"],
    capture_output=True, text=True
)
# Exportar directamente desde este notebook
import json, numpy as np

def serial(obj):
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    raise TypeError

pct2 = calcular_porcentajes_decision(df_e["Decision_Semaforo"])

# Construir JSON de resultados (ver src/features.py para definición completa)
hist_c, hist_e = np.histogram(df_e["Score_Confiabilidad"], bins=range(0,105,5))

tl2 = df_e.groupby("minuto").agg(
    total=("ID_Publicacion","count"),
    cancelar=("Decision_Semaforo",lambda x:(x=="Cancelar").sum()),
    bajar=("Decision_Semaforo",lambda x:(x=="Bajar el video").sum()),
    mantener=("Decision_Semaforo",lambda x:(x=="Mantener").sum()),
    score_prom=("Score_Confiabilidad","mean"),
).round(2).reset_index()

top_j = (df_e.groupby("Usuario_Handle").agg(
    is2=("Indice_Sospecha","mean"), sc2=("Score_Confiabilidad","mean"),
    n2=("ID_Publicacion","count"),
    p2=("Perfil_Usuario",lambda x:x.mode()[0]),
    pl2=("Plataforma",lambda x:x.mode()[0]),
).query("n2>=2").sort_values("is2",ascending=False).head(30).round(2).reset_index())

muestras2 = {}
for zona in ["Cancelar","Bajar el video","Mantener"]:
    sub = df_e[df_e["Decision_Semaforo"]==zona].sort_values("Score_Confiabilidad").head(5)
    muestras2[zona] = [{"id":int(r.ID_Publicacion),"usuario":r.Usuario_Handle,
        "plataforma":r.Plataforma,"tipo":r.Tipo_Publicacion,
        "texto":str(r.Texto_Publicacion)[:200],
        "score":float(r.Score_Confiabilidad),"indice_sospecha":float(r.Indice_Sospecha),
        "peso_prueba":float(r.Peso_Prueba),"tasa_coordinacion":float(r.Tasa_Coordinacion)}
        for _, r in sub.iterrows()]

plat_s = df_e.groupby("Plataforma").agg(total=("ID_Publicacion","count"),score_promedio=("Score_Confiabilidad","mean"),indice_sospecha=("Indice_Sospecha","mean")).round(2)
tipo_s = df_e.groupby("Tipo_Publicacion").agg(total=("ID_Publicacion","count"),score_promedio=("Score_Confiabilidad","mean"),peso_prueba=("Peso_Prueba","mean")).round(2)
perf_s = df_e.groupby("Perfil_Usuario").agg(total=("ID_Publicacion","count"),score_promedio=("Score_Confiabilidad","mean"),indice_sospecha=("Indice_Sospecha","mean")).round(2)

def df_to_list(d):
    r=[]
    for _, row in d.reset_index().iterrows():
        rr={}
        for k,v in row.items():
            rr[k]=int(v) if isinstance(v,(np.integer,)) else float(v) if isinstance(v,(np.floating,)) else v
        r.append(rr)
    return r

resultado = {
    "resumen":{"total_publicaciones":int(len(df_e)),"periodo_inicio":str(df_e["Fecha_Hora_Publicacion"].min()),
               "periodo_fin":str(df_e["Fecha_Hora_Publicacion"].max()),
               "score_promedio":round(float(df_e["Score_Confiabilidad"].mean()),2),
               "score_mediana":round(float(df_e["Score_Confiabilidad"].median()),2),
               "indice_sospecha_promedio":round(float(df_e["Indice_Sospecha"].mean()),3),"decision":pct2},
    "por_plataforma":df_to_list(plat_s),
    "por_tipo_publicacion":df_to_list(tipo_s),
    "por_perfil_usuario":df_to_list(perf_s),
    "linea_de_tiempo":[{"hora":r["minuto"].strftime("%H:%M"),"total":int(r["total"]),"cancelar":int(r["cancelar"]),"bajar":int(r["bajar"]),"mantener":int(r["mantener"]),"score_prom":float(r["score_prom"])} for _,r in tl2.iterrows()],
    "top_cuentas_sospechosas":[{"usuario":r.Usuario_Handle,"indice_sospecha":float(r.is2),"score_confiabilidad":float(r.sc2),"num_publicaciones":int(r.n2),"perfil":r.p2,"plataforma":r.pl2} for _,r in top_j.iterrows()],
    "distribucion_score":[{"bin_inicio":int(hist_e[i]),"bin_fin":int(hist_e[i+1]),"conteo":int(hist_c[i])} for i in range(len(hist_c))],
    "muestras_por_zona":muestras2,
    "metadata":{"generado_el":"2026-10-13","version":"1.0","fuente":"data/BASE.csv","nota":"Datos sinteticos — fines academicos"},
}

with open("outputs/resultados.json","w",encoding="utf-8") as f:
    json.dump(resultado,f,ensure_ascii=False,indent=2,default=serial)
print("outputs/resultados.json exportado OK")
print(f"Decisión: {pct2['decision_final']} ({pct2['confianza_pct']:.1f}%)")


## 10. Conclusiones

### Hallazgos principales

1. **Campaña orquestada**: el 25.9% de las publicaciones provienen de cuentas nuevas o bots. La mayoría surge en las primeras 2 horas — patrón atípico para un escándalo orgánico.

2. **Evidencia inicial no verificable**: la publicación que dispara el escándalo es una *Transcripción de audio/video* desde una cuenta anónima con solo 3 días de antigüedad (Score = 14.7 — el más bajo del dataset).

3. **Contenido reciclado**: 190 publicaciones (5.9%) usan material idéntico a un caso de 2023 — clásico en campañas de desinformación fabricada.

4. **Score promedio = 48.5/100** (zona amarilla): la evidencia es insuficiente para una cancelación definitiva, pero tampoco para ignorar el caso.

5. **Clustering revela 3 grupos naturales**: alta sospecha (bots coordinados), sospecha media (comentarios anónimos), y baja sospecha (cuentas verificadas / fans antiguos).

### Recomendación

> **BAJAR EL VIDEO — 68.3% de confianza**  
> La plataforma debe suspender el contenido inicial mientras se realiza una verificación independiente. No hay suficiente evidencia verificable para una cancelación definitiva.

### Limitaciones
- Datos sintéticos: los patrones pueden diferir de un caso real.
- Sin acceso a metadatos de red (IPs, grafos de seguidores) para confirmar coordinación.
- Análisis semántico del texto no implementado en esta versión.
